In [4]:
import pandas as pd
import mysql.connector
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Pull data using a robust LEFT JOIN
conn = mysql.connector.connect(
    host="localhost", user="root", password="as123AS@", database="rto_optimization"
)
query = """
    SELECT 
        r.distance_km, 
        f.package_weight_kg, 
        f.delivery_cost, 
        c.delivery_partner, 
        f.weather_condition, 
        f.delivery_mode, 
        f.delivery_status
    FROM fact_shipments f
    JOIN dim_couriers c ON f.partner_id = c.partner_id
    LEFT JOIN dim_routes r ON f.route_id = r.route_id;
"""
df = pd.read_sql(query, conn)
conn.close()

# Handle any floating-point lookup gaps gracefully
if df['distance_km'].isna().any():
    df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())

# 2. FIXED: Case-Insensitive Target Conversion (Catches 'Failed' and 'failed')
df['target'] = df['delivery_status'].str.lower().apply(lambda x: 1 if x == 'failed' else 0)

# 3. Categorical Handling via One-Hot Encoding
features = ['distance_km', 'package_weight_kg', 'delivery_cost', 'delivery_partner', 'weather_condition', 'delivery_mode']
X = pd.get_dummies(df[features], drop_first=True)
y = df['target']

print(f"✅ Extracted Matrix Shape: {X.shape}")
print(f"📊 Real Production Target Distribution: {dict(y.value_counts())}")

C:\Users\neera\AppData\Local\Temp\ipykernel_3412\3230549703.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


✅ Extracted Matrix Shape: (13589, 25)
📊 Real Production Target Distribution: {0: np.int64(12876), 1: np.int64(713)}


In [5]:
# Stratified splitting to keep the minority proportions intact
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Calculate the true class imbalance penalty multiplier
neg_count = sum(y_train == 0)
pos_count = sum(y_train == 1)
scale_multiplier = neg_count / pos_count if pos_count > 0 else 1

# Train our champion model
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_multiplier,
    eval_metric="aucpr",  # Optimize for Precision-Recall frontier
    random_state=42,
    learning_rate=0.1,
    max_depth=5
)

xgb_model.fit(X_train, y_train)
print(f"🌲 Tree Ensemble trained successfully! Imbalance penalty weight: {round(scale_multiplier, 2)}x")

🌲 Tree Ensemble trained successfully! Imbalance penalty weight: 18.07x


In [6]:
# Generate true predictive probabilities
y_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate the classification report at standard threshold
y_pred = (y_probs >= 0.5).astype(int)
print("\n--- Real Production Model Evaluation ---")
print(classification_report(y_test, y_pred, zero_division=0))


--- Real Production Model Evaluation ---
              precision    recall  f1-score   support

           0       0.98      0.76      0.86      2575
           1       0.15      0.77      0.25       143

    accuracy                           0.76      2718
   macro avg       0.57      0.76      0.55      2718
weighted avg       0.94      0.76      0.82      2718

